# V7: five-fold YOLO-only segmentation, 10 epochs maximum

This standalone Kaggle notebook evaluates a YOLO11s-seg detector with five
observation-day-grouped outer folds. Each fold trains for **up to 10 epochs**;
there are **no wall-clock time limits**. Ultralytics may stop earlier if its
validation early-stopping criterion is met. Inner tuning selects confidence and
minimum-area settings, and the five completed models are fused for test export.

V7 removes the crop/refiner pipeline entirely. It does not resume or reuse V5/V6
checkpoints. Run with the competition dataset, Internet enabled for pretrained
weights, and a GPU accelerator. Training is resumable after an interruption.
The final cells prepare and verify a CSV, then submit it with the Kaggle CLI. Configure Kaggle API credentials before running the submission cell.


In [ ]:
%pip install -q 'ultralytics==8.4.211' pycocotools scipy kaggle


In [ ]:
import hashlib
import json
import random
import subprocess
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pycocotools.mask as mask_utils
import torch
from scipy.optimize import linear_sum_assignment
from PIL import Image

V7 = dict(
    weights="yolo11s-seg.pt", image_size=1024, batch_size=10, epochs=10,
    patience=10, learning_rate=0.002, workers=2, inference_batch=8,
    tuning_images=32, confidences=[0.001, 0.01, 0.05], min_areas=[64, 256],
)
CFG = dict(seed=2026)
RUN_NAME = "v7_yolo_only_fivefold_10_epochs_seed2026"
SMOKE_TEST = False
DATA_ROOT_OVERRIDE = None
OUTPUT_DIR = (Path("/kaggle/working") if Path("/kaggle/working").is_dir()
              else Path.cwd() / "cv_outputs") / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CV_ROOT = OUTPUT_DIR
COMPETITION = "filament-segmentation-2026"
DATASET_NAME = "MAGFiLO_1.0_Kaggle_2026"
ANNOTATION_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
FULL_SHAPE = (2048, 2048)
FOLDS_TO_RUN = [0, 1, 2, 3, 4]
INNER_TUNE_FRACTION = 0.1
COLLECT_ROOTS = []  # Add matching fold directories downloaded from earlier runs.
if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    YOLO_TRAIN_DEVICE = [0, 1]
    YOLO_INFERENCE_DEVICE = 0
elif SMOKE_TEST:
    YOLO_TRAIN_DEVICE = YOLO_INFERENCE_DEVICE = "cpu"
else:
    raise RuntimeError("V7 requires two visible GPUs (Kaggle T4x2) for batch size 10.")

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["seed"])
if not torch.cuda.is_available() and not SMOKE_TEST:
    print("WARNING: Full training requires a GPU; enable a Kaggle GPU before running folds.")
print("Training devices:", YOLO_TRAIN_DEVICE, "inference device:", YOLO_INFERENCE_DEVICE,
      "outputs:", CV_ROOT)


In [ ]:
candidates = ([Path(DATA_ROOT_OVERRIDE)] if DATA_ROOT_OVERRIDE else [
    Path('/kaggle/input/competitions') / COMPETITION / DATASET_NAME,
    Path('/kaggle/input') / COMPETITION / DATASET_NAME,
    Path.cwd() / 'data' / DATASET_NAME, Path.cwd() / DATASET_NAME])
if DATA_ROOT_OVERRIDE is None and Path('/kaggle/input').is_dir():
    candidates += sorted(Path('/kaggle/input').glob(f'*/{DATASET_NAME}'))
    candidates += sorted(Path('/kaggle/input').glob(f'*/*/{DATASET_NAME}'))
valid_roots = [p for p in candidates if (p / 'train' / ANNOTATION_NAME).is_file()
               and (p / 'test' / 'test_images').is_dir()]
if not valid_roots:
    raise FileNotFoundError('Attach the competition data or set DATA_ROOT_OVERRIDE to its dataset root.')
DATA_ROOT = valid_roots[0]
train_paths = sorted((DATA_ROOT / 'train' / 'train_images').glob('*.jpeg'))
test_paths = sorted((DATA_ROOT / 'test' / 'test_images').glob('*.jpeg'))
if not train_paths or not test_paths:
    raise FileNotFoundError('Train and test must both contain JPEG files.')
with (DATA_ROOT / 'train' / ANNOTATION_NAME).open() as stream:
    coco = json.load(stream)
records = coco['images']
if len({r['id'] for r in records}) != len(records):
    raise ValueError('Duplicate COCO image IDs.')
records_by_name = defaultdict(list)
annotations_by_id = defaultdict(list)
for record in records:
    if (record['height'], record['width']) != FULL_SHAPE:
        raise ValueError(f'Unexpected dimensions for {record["id"]}')
    records_by_name[record['file_name']].append(record)
for ann in coco['annotations']:
    if ann.get('iscrowd', 0) != 0:
        raise ValueError('Local PQ assumes individual non-crowd filaments.')
    annotations_by_id[ann['image_id']].append(ann)
known_ids = {r['id'] for r in records}
if set(annotations_by_id) - known_ids:
    raise ValueError('Orphan annotation image IDs.')
if len({a['id'] for a in coco['annotations']}) != len(coco['annotations']):
    raise ValueError('Duplicate filament annotation IDs.')
train_lookup = {p.name: p for p in train_paths}
if set(records_by_name) - set(train_lookup):
    raise FileNotFoundError('Annotations reference missing training files.')
if set(train_lookup) & {p.name for p in test_paths}:
    raise ValueError('Train/test filenames overlap.')

annotation_sha256 = hashlib.sha256((DATA_ROOT / 'train' / ANNOTATION_NAME).read_bytes()).hexdigest()
print('Dataset:', len(train_paths), 'labeled physical images;', len(test_paths), 'unlabeled test images')


In [ ]:
def ann_rle(annotation):
    segmentation = annotation["segmentation"]
    if isinstance(segmentation, list):
        if not segmentation or any(len(p) < 6 or len(p) % 2 for p in segmentation):
            raise ValueError(f"Invalid polygon: {annotation['id']}")
        rle = mask_utils.merge(mask_utils.frPyObjects(segmentation, *FULL_SHAPE))
    elif isinstance(segmentation, dict):
        if list(segmentation["size"]) != list(FULL_SHAPE):
            raise ValueError("RLE dimensions do not match the image.")
        if isinstance(segmentation["counts"], list):
            rle = mask_utils.frPyObjects(segmentation, *FULL_SHAPE)
        else:
            counts = segmentation["counts"]
            rle = dict(size=list(FULL_SHAPE), counts=counts.encode("ascii") if isinstance(counts, str) else counts)
    else:
        raise TypeError(f"Unsupported segmentation: {annotation['id']}")
    if mask_utils.area(rle) <= 0:
        raise ValueError(f"Empty ground-truth filament: {annotation['id']}")
    return rle


def gt_rles(image_id):
    return [ann_rle(annotation) for annotation in annotations_by_id[image_id]]


def local_scores(predicted, truth):
    n_pred, n_true = len(predicted), len(truth)
    overlaps = (mask_utils.iou(predicted, truth, [0] * n_true)
                if n_pred and n_true else np.zeros((n_pred, n_true)))
    matched = []
    if n_pred and n_true:
        valid = overlaps > 0.5
        rewards = valid * (1 + overlaps / (min(n_pred, n_true) + 1))
        rows, columns = linear_sum_assignment(-rewards)
        matched = [float(overlaps[r, c]) for r, c in zip(rows, columns) if valid[r, c]]
    tp = len(matched)
    fp, fn = n_pred - tp, n_true - tp
    denominator = tp + 0.5 * fp + 0.5 * fn
    pq = sum(matched) / denominator if denominator else 1.0
    pred_union = mask_utils.decode(mask_utils.merge(predicted)) if predicted else np.zeros(FULL_SHAPE, dtype=np.uint8)
    true_union = mask_utils.decode(mask_utils.merge(truth)) if truth else np.zeros(FULL_SHAPE, dtype=np.uint8)
    intersection = np.count_nonzero(pred_union.astype(bool) & true_union.astype(bool))
    union = np.count_nonzero(pred_union.astype(bool) | true_union.astype(bool))
    pixels = np.count_nonzero(pred_union) + np.count_nonzero(true_union)
    return dict(pq=pq, dice=2 * intersection / pixels if pixels else 1.0,
                iou=intersection / union if union else 1.0, tp=tp, fp=fp, fn=fn)


In [ ]:
import sys
from pathlib import Path

runtime_dir = CV_ROOT / "_runtime"
runtime_dir.mkdir(parents=True, exist_ok=True)
(runtime_dir / "grouped_yolo_cv.py").write_text('"""Day-grouped outer CV with inner tuning for the standalone YOLO/refiner notebook."""\n\nimport hashlib\nimport csv\nimport json\nimport math\nimport random\nfrom collections import defaultdict\nfrom datetime import datetime\nfrom pathlib import Path\n\nimport numpy as np\nimport pycocotools.mask as mask_utils\nfrom scipy.optimize import linear_sum_assignment\n\n\ndef cv_digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()\n\n\ndef make_day_folds(names, seed=2026, folds=5, inner_fraction=0.1):\n    names = sorted(names)\n    if len(set(names)) != len(names) or folds != 5 or not 0 < inner_fraction < 1:\n        raise ValueError("Use unique physical filenames, five folds, and 0 < inner_fraction < 1.")\n    by_day = defaultdict(list)\n    for name in names:\n        datetime.strptime(name[:8], "%Y%m%d")\n        by_day[name[:8]].append(name)\n    if len(by_day) < 10:\n        raise ValueError("Five-fold CV needs at least ten observation days.")\n    days = sorted(by_day)\n    random.Random(seed).shuffle(days)\n    # Largest physical-image groups first; seeded order breaks equal-size ties.\n    days.sort(key=lambda day: -len(by_day[day]))\n    outer_days, sizes = [[] for _ in range(folds)], [0] * folds\n    for day in days:\n        fold = min(range(folds), key=lambda index: (sizes[index], index))\n        outer_days[fold].append(day)\n        sizes[fold] += len(by_day[day])\n    plans = []\n    for fold in range(folds):\n        remaining = sorted(set(days) - set(outer_days[fold]))\n        random.Random(seed + fold + 1).shuffle(remaining)\n        n_inner = max(1, round(len(remaining) * inner_fraction))\n        if n_inner >= len(remaining):\n            raise ValueError("Inner tuning leaves no training days.")\n        groups = dict(\n            train=set(remaining[n_inner:]), tune=set(remaining[:n_inner]),\n            holdout=set(outer_days[fold]),\n        )\n        split = {key: sorted(name for day in selected for name in by_day[day])\n                 for key, selected in groups.items()}\n        if any(not selected for selected in split.values()):\n            raise ValueError("Every CV partition must be nonempty.")\n        if set.union(*groups.values()) != set(by_day):\n            raise ValueError("CV partitions do not cover the labeled dataset.")\n        if any(groups[a] & groups[b] for a, b in\n               [("train", "tune"), ("train", "holdout"), ("tune", "holdout")]):\n            raise ValueError("Observation-day leakage.")\n        plans.append(dict(fold=fold, splits=split))\n    outer = [name for plan in plans for name in plan["splits"]["holdout"]]\n    if len(outer) != len(names) or set(outer) != set(names):\n        raise ValueError("Each physical image must appear in exactly one outer fold.")\n    return plans\n\n\ndef save_cv_plan(path, plan):\n    path = Path(path)\n    if path.exists():\n        if json.loads(path.read_text()) != plan:\n            raise ValueError("Existing CV plan differs. Choose a new CV_RUN_NAME.")\n    else:\n        path.write_text(json.dumps(plan, indent=2, allow_nan=False))\n\n\ndef fold_records(plan, records_by_name):\n    return {phase: [record for name in names for record in records_by_name[name]]\n            for phase, names in plan["splits"].items()}\n\n\ndef validate_fold_result(result, plan, plan_hash):\n    if (result["plan_sha256"] != plan_hash or result["fold"] != plan["fold"]\n            or result["status"] != "complete"):\n        raise ValueError("Fold result provenance/status mismatch.")\n    rows = result["outer_rows"]\n    expected = plan["splits"]["holdout"]\n    actual = [row["filename"] for row in rows]\n    if len(actual) != len(set(actual)) or set(actual) != set(expected):\n        raise ValueError("Outer rows must cover exactly this fold\'s physical images.")\n    if not result["checkpoint_hashes"] or not result["selected_parameters"]:\n        raise ValueError("Missing selected model provenance.")\n    for row in rows:\n        if row["annotator_sets"] < 1:\n            raise ValueError("Missing outer annotator evaluation.")\n        for key in ["pq", "dice", "iou"]:\n            if not isinstance(row[key], (int, float)) or not math.isfinite(row[key]) or not 0 <= row[key] <= 1:\n                raise ValueError("Invalid outer metric.")\n    mean = float(np.mean([row["pq"] for row in rows]))\n    if not math.isclose(mean, result["outer_mean"]["pq"], abs_tol=1e-12):\n        raise ValueError("Outer PQ summary disagrees with per-image rows.")\n\n\ndef summarize_folds(results, plans, plan_hash):\n    by_fold = {}\n    for result in results:\n        fold = result["fold"]\n        if fold not in range(5) or fold in by_fold:\n            raise ValueError("Unknown or duplicate completed fold.")\n        validate_fold_result(result, plans[fold], plan_hash)\n        by_fold[fold] = result\n    complete = len(by_fold) == 5\n    summary = dict(\n        plan_sha256=plan_hash, complete=complete, completed_folds=sorted(by_fold),\n        missing_folds=sorted(set(range(5)) - set(by_fold)),\n        status="complete" if complete else "incomplete: no five-fold aggregate reported",\n    )\n    if complete:\n        fold_pq = [by_fold[fold]["outer_mean"]["pq"] for fold in range(5)]\n        rows = [row for fold in range(5) for row in by_fold[fold]["outer_rows"]]\n        summary.update(\n            fold_mean_pq=float(np.mean(fold_pq)),\n            fold_sample_sd_pq=float(np.std(fold_pq, ddof=1)),\n            pooled_oof_mean_pq=float(np.mean([row["pq"] for row in rows])),\n            physical_images=len(rows),\n            note="Per-image metrics average annotators first. Fold SD is descriptive, not a confidence interval. "\n                 "These are reused labeled data, not a new untouched final test or leaderboard score.",\n        )\n    return summary\n\n\ndef collect_fold_results(roots, plans, plan_hash):\n    results = {}\n    for root in roots:\n        for path in sorted(Path(root).rglob("fold_result.json")):\n            result = json.loads(path.read_text())\n            fold = result["fold"]\n            if fold not in range(5):\n                raise ValueError(f"Unknown fold in {path}")\n            validate_fold_result(result, plans[fold], plan_hash)\n            if fold in results and results[fold] != result:\n                raise ValueError(f"Conflicting results for fold {fold}; do not select by outer score.")\n            results[fold] = result\n    return [results[fold] for fold in sorted(results)]\n\n\ndef evaluate_outer(names, parameters, namespace, detector, refiner):\n    rows = []\n    for name in namespace["tqdm"](names, desc="Outer evaluation: frozen parameters"):\n        proposals = namespace["v4_proposals"](\n            namespace["train_lookup"][name], detector, refiner, parameters["confidence"],\n        )\n        predicted, fallbacks = namespace["v4_instances"](proposals, parameters)\n        namespace["validate_disjoint"](predicted, namespace["FULL_SHAPE"])\n        scores = [namespace["local_scores"](predicted, namespace["gt_rles"](record["id"]))\n                  for record in namespace["records_by_name"][name]]\n        if not scores:\n            raise ValueError("No annotator set for outer image.")\n        rows.append(dict(\n            filename=name, annotator_sets=len(scores), instances=len(predicted),\n            refiner_fallbacks=fallbacks,\n            **{key: float(np.mean([score[key] for score in scores])) for key in scores[0]},\n        ))\n    if not rows:\n        raise ValueError("No outer images.")\n    means = {key: float(np.mean([row[key] for row in rows]))\n             for key in rows[0] if key not in ["filename", "annotator_sets"]}\n    return means, rows\n\n\ndef run_cv_fold(namespace, plan, shared_plan, root):\n    """Run one isolated fold; outer labels never enter training or calibration."""\n    plan_hash = cv_digest(shared_plan)\n    directory = Path(root) / f"fold_{plan[\'fold\']}"\n    directory.mkdir(parents=True, exist_ok=True)\n    result_path = directory / "fold_result.json"\n    if result_path.exists():\n        result = json.loads(result_path.read_text())\n        validate_fold_result(result, plan, plan_hash)\n        print("Reusing completed outer evaluation:", result_path)\n        return result\n    namespace["OUTPUT_DIR"] = directory\n    namespace["split_names"] = plan["splits"]\n    namespace["split_records"] = fold_records(plan, namespace["records_by_name"])\n    namespace["manifest"] = dict(\n        seed=namespace["CFG"]["seed"], grouping="observation-day",\n        annotation_sha256=namespace["annotation_sha256"], splits=plan["splits"],\n    )\n    namespace["write_json"](directory / "split_manifest.json", namespace["manifest"])\n    for flag, files in [\n        ("RUN_YOLO_TRAINING", ["yolo_best.pt", "yolo_provenance.json"]),\n        ("RUN_REFINER_TRAINING", ["refiner_best.pt", "refiner_runtime.json"]),\n    ]:\n        existing = [(directory / name).exists() for name in files]\n        if any(existing) and not all(existing):\n            raise ValueError(f"Incomplete checkpoint evidence in {directory}; recover it explicitly.")\n        namespace[flag] = not all(existing)\n    namespace["RUN_V4_CALIBRATION"] = not (directory / "v4_calibration.json").exists()\n    detector = refiner = None\n    try:\n        detector = namespace["v4_train_yolo"]()\n        refiner = namespace["v4_train_refiner"]()\n        calibration = namespace["v4_calibrate"](detector, refiner)\n        means, rows = evaluate_outer(\n            plan["splits"]["holdout"], calibration["selected"]["parameters"],\n            namespace, detector, refiner,\n        )\n        result = dict(\n            fold=plan["fold"], plan_sha256=plan_hash, status="complete",\n            smoke_test=shared_plan["smoke_test"],\n            checkpoint_hashes=calibration["hashes"],\n            selected_parameters=calibration["selected"]["parameters"],\n            inner_selected_scores=calibration["selected"]["scores"],\n            outer_mean=means, outer_rows=rows,\n        )\n        validate_fold_result(result, plan, plan_hash)\n        namespace["write_json"](directory / "fold_result.json", result)\n        return result\n    finally:\n        del detector, refiner\n        namespace["gc"].collect()\n        if namespace["torch"].cuda.is_available():\n            namespace["torch"].cuda.empty_cache()\n\n\ndef export_cv_submission(namespace, plan, shared_plan, root):\n    """Export a predeclared fold; never select a model by outer score."""\n    directory = Path(root) / f"fold_{plan[\'fold\']}"\n    result = json.loads((directory / "fold_result.json").read_text())\n    validate_fold_result(result, plan, cv_digest(shared_plan))\n    namespace.update(\n        OUTPUT_DIR=directory, split_names=plan["splits"],\n        split_records=fold_records(plan, namespace["records_by_name"]),\n        RUN_YOLO_TRAINING=False, RUN_REFINER_TRAINING=False, RUN_V4_CALIBRATION=False,\n    )\n    detector = refiner = None\n    try:\n        detector = namespace["v4_train_yolo"]()\n        refiner = namespace["v4_train_refiner"]()\n        calibration = namespace["v4_calibrate"](detector, refiner)\n        if calibration["hashes"] != result["checkpoint_hashes"]:\n            raise ValueError("Submission checkpoints differ from the completed fold.")\n        return namespace["v4_export"](detector, refiner, calibration)\n    finally:\n        del detector, refiner\n        namespace["gc"].collect()\n        if namespace["torch"].cuda.is_available():\n            namespace["torch"].cuda.empty_cache()\n\n\nENSEMBLE_POLICY = dict(\n    method="equal-fold-instance-majority-v1", match_iou=0.5,\n    required_fold_votes=3, pixel_votes=3, min_area=20,\n)\n\n\ndef fuse_fold_instances(fold_instances, shape):\n    """Match at most one instance per fold per cluster, then vote across five folds."""\n    if len(fold_instances) != 5:\n        raise ValueError("Ensembling requires predictions from exactly five folds.")\n    clusters = []\n    for instances in fold_instances:\n        for rle in instances:\n            if tuple(rle["size"]) != tuple(shape) or not mask_utils.area(rle):\n                raise ValueError("Invalid ensemble instance shape or empty mask.")\n        if not instances:\n            continue\n        representatives = []\n        for cluster in clusters:\n            votes = sum((mask_utils.decode(rle).astype(np.uint8) for rle in cluster),\n                        np.zeros(shape, dtype=np.uint8))\n            representatives.append(mask_utils.encode(np.asfortranarray(\n                (votes * 2 >= len(cluster)).astype(np.uint8))))\n        matched = set()\n        if representatives:\n            overlaps = mask_utils.iou(representatives, instances, [0] * len(instances))\n            eligible = overlaps >= ENSEMBLE_POLICY["match_iou"]\n            # Maximize valid match count first, then IoU; invalid pairs cannot steal matches.\n            weights = eligible * (min(overlaps.shape) + 1 + overlaps)\n            rows, columns = linear_sum_assignment(-weights)\n            for row, column in zip(rows, columns):\n                if eligible[row, column]:\n                    clusters[row].append(instances[column])\n                    matched.add(column)\n        clusters.extend([rle] for index, rle in enumerate(instances) if index not in matched)\n    fused = []\n    for cluster in clusters:\n        if len(cluster) < ENSEMBLE_POLICY["required_fold_votes"]:\n            continue\n        votes = sum((mask_utils.decode(rle).astype(np.uint8) for rle in cluster),\n                    np.zeros(shape, dtype=np.uint8))\n        mask = (votes >= ENSEMBLE_POLICY["pixel_votes"]).astype(np.uint8)\n        if int(mask.sum()) >= ENSEMBLE_POLICY["min_area"]:\n            fused.append((len(cluster), int(mask.sum()),\n                          mask_utils.encode(np.asfortranarray(mask))))\n    fused.sort(key=lambda item: (-item[0], -item[1]))\n    return [rle for _, _, rle in fused]\n\n\ndef export_cv_ensemble(namespace, plans, shared_plan, root):\n    """Reload all five models sequentially and export one validated voted submission."""\n    if shared_plan["smoke_test"] or namespace["SMOKE_TEST"]:\n        raise ValueError("Smoke runs cannot produce competition exports.")\n    if sorted(plan["fold"] for plan in plans) != list(range(5)):\n        raise ValueError("All five unique fold plans are required.")\n    plans = sorted(plans, key=lambda plan: plan["fold"])\n    root = Path(root)\n    results = []\n    for plan in plans:\n        result = json.loads((root / f"fold_{plan[\'fold\']}" / "fold_result.json").read_text())\n        validate_fold_result(result, plan, cv_digest(shared_plan))\n        if result.get("smoke_test", False):\n            raise ValueError("Cannot ensemble smoke checkpoints.")\n        results.append(result)\n    test_paths = namespace["test_paths"]\n    if not test_paths or len({path.stem for path in test_paths}) != len(test_paths):\n        raise ValueError("Test image IDs must be nonempty and unique.")\n    prediction_paths = []\n    for plan, result in zip(plans, results):\n        directory = root / f"fold_{plan[\'fold\']}"\n        namespace.update(\n            OUTPUT_DIR=directory, split_names=plan["splits"],\n            split_records=fold_records(plan, namespace["records_by_name"]),\n            RUN_YOLO_TRAINING=False, RUN_REFINER_TRAINING=False, RUN_V4_CALIBRATION=False,\n        )\n        detector = refiner = None\n        try:\n            detector = namespace["v4_train_yolo"]()\n            refiner = namespace["v4_train_refiner"]()\n            calibration = namespace["v4_calibrate"](detector, refiner)\n            if (calibration["hashes"] != result["checkpoint_hashes"]\n                    or calibration["selected"]["parameters"] != result["selected_parameters"]):\n                raise ValueError("Ensemble checkpoints/calibration differ from completed fold.")\n            parameters = calibration["selected"]["parameters"]\n            predictions = {}\n            for path in namespace["tqdm"](test_paths, desc=f"Ensemble fold {plan[\'fold\']} test"):\n                proposals = namespace["v4_proposals"](\n                    path, detector, refiner, parameters["confidence"])\n                instances, _ = namespace["v4_instances"](proposals, parameters)\n                namespace["validate_disjoint"](instances, namespace["FULL_SHAPE"])\n                predictions[path.stem] = [\n                    dict(size=list(rle["size"]), counts=rle["counts"].decode("ascii"))\n                    for rle in instances\n                ]\n            prediction_path = directory / "ensemble_predictions.json"\n            namespace["write_json"](prediction_path, predictions)\n            prediction_paths.append(prediction_path)\n        finally:\n            del detector, refiner\n            namespace["gc"].collect()\n            if namespace["torch"].cuda.is_available():\n                namespace["torch"].cuda.empty_cache()\n    predictions = [json.loads(path.read_text()) for path in prediction_paths]\n    rows, image_audits = [], []\n    for path in test_paths:\n        per_fold = [\n            [dict(size=rle["size"], counts=rle["counts"].encode("ascii"))\n             for rle in fold[path.stem]] for fold in predictions\n        ]\n        instances = fuse_fold_instances(per_fold, namespace["FULL_SHAPE"])\n        instances, ownership = namespace["disjoint_instances"](\n            instances, namespace["FULL_SHAPE"], ENSEMBLE_POLICY["min_area"])\n        namespace["validate_disjoint"](instances, namespace["FULL_SHAPE"])\n        for rle in instances:\n            if mask_utils.encode(np.asfortranarray(mask_utils.decode(rle)))["counts"] != rle["counts"]:\n                raise ValueError("Ensemble RLE roundtrip failed.")\n        image_audits.append(dict(\n            filename=path.name, fold_instances=[len(items) for items in per_fold],\n            instances=len(instances), ownership=ownership,\n        ))\n        rows.extend(dict(filament_id=f"{path.stem}_{index}",\n                         segmentation_rle=rle["counts"].decode("ascii"))\n                    for index, rle in enumerate(instances, 1))\n    audit = namespace["validate_submission_rows"](\n        rows, {path.stem for path in test_paths}, namespace["FULL_SHAPE"])\n    destination = root / "submission.csv"\n    temporary = root / "submission.ensemble.tmp.csv"\n    fields = ["filament_id", "segmentation_rle"]\n    try:\n        with temporary.open("w", newline="") as stream:\n            writer = csv.DictWriter(stream, fieldnames=fields)\n            writer.writeheader()\n            writer.writerows(rows)\n        with temporary.open(newline="") as stream:\n            reader = csv.DictReader(stream)\n            if reader.fieldnames != fields or list(reader) != rows:\n                raise ValueError("Ensemble CSV readback mismatch.")\n        namespace["write_json"](root / "ensemble_audit.json", dict(\n            policy=ENSEMBLE_POLICY, plan_sha256=cv_digest(shared_plan),\n            folds=[dict(fold=r["fold"], hashes=r["checkpoint_hashes"],\n                        parameters=r["selected_parameters"]) for r in results],\n            validation=audit, images=image_audits,\n            submission_sha256=hashlib.sha256(temporary.read_bytes()).hexdigest(),\n        ))\n        temporary.replace(destination)\n    finally:\n        if temporary.exists():\n            temporary.unlink()\n    kaggle_root = Path("/kaggle/working")\n    if kaggle_root.is_dir() and destination.resolve() != (kaggle_root / "submission.csv").resolve():\n        namespace["shutil"].copy2(destination, kaggle_root / "submission.csv")\n    namespace["OUTPUT_DIR"] = root\n    if audit["zero_detection_images"]:\n        print("WARNING: zero-detection images omitted, no invented masks:",\n              audit["zero_detection_images"])\n    return destination\n')
(runtime_dir / "submission_masks.py").write_text('"""Disjoint instance masks and an offline repair command for saved V3 CSVs."""\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nimport pycocotools.mask as mask_utils\n\n\nMASK_POLICY = "first-mask-wins-v1"\nSUBMISSION_FIELDS = ["filament_id", "segmentation_rle"]\n\n\ndef decode_instance(rle, shape):\n    if tuple(rle["size"]) != tuple(shape):\n        raise ValueError(f"Instance shape {rle[\'size\']} does not match {shape}.")\n    mask = mask_utils.decode(rle)\n    if mask.shape != tuple(shape) or not np.isin(mask, [0, 1]).all() or not mask.any():\n        raise ValueError("An instance must be a nonempty binary mask.")\n    return mask.astype(bool)\n\n\ndef disjoint_instances(instances, shape=(2048, 2048), min_area=1):\n    """Keep earlier masks unchanged; later masks may claim only unowned pixels."""\n    if min_area < 1:\n        raise ValueError("min_area must be positive.")\n    occupied = np.zeros(shape, dtype=bool)\n    result, kept_indices, dropped_indices = [], [], []\n    overlap_pixels = 0\n    for index, rle in enumerate(instances):\n        mask = decode_instance(rle, shape)\n        overlap_pixels += int(np.count_nonzero(mask & occupied))\n        mask &= ~occupied\n        if np.count_nonzero(mask) < min_area:\n            dropped_indices.append(index)\n            continue\n        occupied |= mask\n        result.append(mask_utils.encode(np.asfortranarray(mask.astype(np.uint8))))\n        kept_indices.append(index)\n    return result, dict(\n        policy=MASK_POLICY, overlap_pixels_removed=overlap_pixels,\n        kept_indices=kept_indices, dropped_indices=dropped_indices,\n    )\n\n\ndef validate_disjoint(instances, shape=(2048, 2048)):\n    occupied = np.zeros(shape, dtype=bool)\n    for index, rle in enumerate(instances):\n        mask = decode_instance(rle, shape)\n        if np.any(occupied & mask):\n            raise ValueError(f"Instance {index} overlaps an earlier instance.")\n        occupied |= mask\n\n\ndef submission_groups(rows, test_stems, shape):\n    groups, seen = defaultdict(list), set()\n    for row in rows:\n        if set(row) != set(SUBMISSION_FIELDS):\n            raise ValueError("CSV must have exactly the two competition columns.")\n        identity = row["filament_id"]\n        stem, separator, number = identity.rpartition("_")\n        if (not separator or not number.isdigit() or int(number) < 1\n                or stem not in test_stems or identity in seen):\n            raise ValueError(f"Unknown, invalid, or duplicate filament_id: {identity}")\n        seen.add(identity)\n        rle = dict(size=list(shape), counts=row["segmentation_rle"].encode("ascii"))\n        groups[stem].append((row, rle))\n    if not groups:\n        raise ValueError("Submission has no predicted instances.")\n    return groups\n\n\ndef validate_submission_rows(rows, test_stems, shape=(2048, 2048)):\n    groups = submission_groups(rows, test_stems, shape)\n    for stem, entries in groups.items():\n        try:\n            validate_disjoint([rle for _, rle in entries], shape)\n        except ValueError as error:\n            raise ValueError(f"{stem}: {error}") from error\n    return dict(\n        rows=len(rows), images_with_predictions=len(groups),\n        zero_detection_images=sorted(set(test_stems) - set(groups)),\n        overlapping_pixels=0,\n    )\n\n\ndef repair_submission(source, destination, test_stems, min_area=20, shape=(2048, 2048)):\n    source, destination = Path(source), Path(destination)\n    audit_path = destination.with_suffix(".audit.json")\n    if source.resolve() == destination.resolve():\n        raise ValueError("Repair must not overwrite the original submission.")\n    if destination.exists() or audit_path.exists():\n        raise FileExistsError("Choose a new repair output path; existing evidence is preserved.")\n    with source.open(newline="") as stream:\n        reader = csv.DictReader(stream)\n        if reader.fieldnames != SUBMISSION_FIELDS:\n            raise ValueError("Unexpected submission columns or column order.")\n        original_rows = list(reader)\n    groups = submission_groups(original_rows, test_stems, shape)\n    repaired_by_id, image_audits = {}, []\n    for stem, entries in groups.items():\n        masks, audit = disjoint_instances([rle for _, rle in entries], shape, min_area)\n        for index, rle in zip(audit["kept_indices"], masks):\n            identity = entries[index][0]["filament_id"]\n            repaired_by_id[identity] = dict(\n                filament_id=identity, segmentation_rle=rle["counts"].decode("ascii"),\n            )\n        image_audits.append(dict(\n            image=stem, overlap_pixels_removed=audit["overlap_pixels_removed"],\n            input_instances=len(entries), output_instances=len(masks),\n            dropped_ids=[entries[i][0]["filament_id"] for i in audit["dropped_indices"]],\n        ))\n    rows = [repaired_by_id[r["filament_id"]] for r in original_rows\n            if r["filament_id"] in repaired_by_id]\n    validation = validate_submission_rows(rows, test_stems, shape)\n    temporary = destination.with_suffix(".tmp.csv")\n    if temporary.exists():\n        raise FileExistsError(f"Temporary file already exists: {temporary}")\n    try:\n        with temporary.open("x", newline="") as stream:\n            writer = csv.DictWriter(stream, fieldnames=SUBMISSION_FIELDS)\n            writer.writeheader()\n            writer.writerows(rows)\n        with temporary.open(newline="") as stream:\n            reader = csv.DictReader(stream)\n            if reader.fieldnames != SUBMISSION_FIELDS or list(reader) != rows:\n                raise ValueError("Repaired CSV readback mismatch.")\n        report = dict(\n            policy=MASK_POLICY, min_area=min_area, validation=validation,\n            input_csv=str(source), input_sha256=hashlib.sha256(source.read_bytes()).hexdigest(),\n            output_sha256=hashlib.sha256(temporary.read_bytes()).hexdigest(),\n            input_rows=len(original_rows), dropped_rows=len(original_rows) - len(rows),\n            images_with_overlap=sum(a["overlap_pixels_removed"] > 0 for a in image_audits),\n            overlap_pixels_removed=sum(a["overlap_pixels_removed"] for a in image_audits),\n            images=image_audits,\n            note="Offline repair only. No retraining, recalibration, or submission. "\n                 "Previous tuning scores do not measure these repaired masks.",\n        )\n        with audit_path.open("x") as stream:\n            stream.write(json.dumps(report, indent=2, allow_nan=False))\n        temporary.replace(destination)\n    finally:\n        if temporary.exists():\n            temporary.unlink()\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("source", type=Path)\n    parser.add_argument("destination", type=Path)\n    parser.add_argument("--test-dir", type=Path, required=True)\n    parser.add_argument("--min-area", type=int, default=20)\n    args = parser.parse_args()\n    test_stems = {p.stem for p in args.test_dir.rglob("*.jpeg")}\n    if not test_stems:\n        parser.error("--test-dir contains no test JPEGs.")\n    report = repair_submission(args.source, args.destination, test_stems, args.min_area)\n    print(json.dumps({key: value for key, value in report.items() if key != "images"}, indent=2))\n    print("REPAIRED CSV:", args.destination, "No competition upload performed.")\n\n\nif __name__ == "__main__":\n    main()\n')
(runtime_dir / "v7_yolo_runtime.py").write_text('"""YOLO-only five-fold execution, resumable batch inference, and audited export."""\n\nimport csv\nimport gc\nimport hashlib\nimport json\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pycocotools.mask as mask_utils\nimport torch\nimport yaml\nfrom ultralytics import YOLO\n\nfrom grouped_yolo_cv import (\n    ENSEMBLE_POLICY, cv_digest, fold_records, fuse_fold_instances,\n    validate_fold_result,\n)\nfrom submission_masks import (\n    SUBMISSION_FIELDS, disjoint_instances, validate_disjoint, validate_submission_rows,\n)\n\n\ndef file_hash(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False))\n    temporary.replace(path)\n\n\ndef require_identity(path, identity):\n    path = Path(path)\n    if path.exists():\n        if json.loads(path.read_text()) != identity:\n            raise ValueError(f"Provenance mismatch: {path}. Use a new run directory.")\n    else:\n        atomic_json(path, identity)\n\n\ndef prepare_dataset(ns, plan, directory):\n    """Keep distinct annotator targets, sharing their physical source JPEGs."""\n    data = directory / "yolo_dataset"\n    records = fold_records(plan, ns["records_by_name"])\n    height, width = ns["FULL_SHAPE"]\n    for phase, label in [("train", "train"), ("tune", "val")]:\n        images, labels = data / "images" / label, data / "labels" / label\n        images.mkdir(parents=True, exist_ok=True)\n        labels.mkdir(parents=True, exist_ok=True)\n        for index, record in enumerate(records[phase]):\n            stem = f\'{Path(record["file_name"]).stem}__annotation{index}\'\n            source = ns["train_lookup"][record["file_name"]].resolve()\n            destination = images / f"{stem}.jpeg"\n            if not destination.exists():\n                destination.symlink_to(source)\n            elif destination.resolve() != source:\n                raise ValueError(f"Training image link differs: {destination}")\n            lines = []\n            for annotation in ns["annotations_by_id"][record["id"]]:\n                polygons = annotation["segmentation"]\n                if not isinstance(polygons, list) or len(polygons) != 1:\n                    raise ValueError("YOLO conversion requires one polygon per instance.")\n                points = np.asarray(polygons[0], dtype=np.float64)\n                if points.size < 6 or points.size % 2 or not np.isfinite(points).all():\n                    raise ValueError(f\'Invalid polygon {annotation["id"]}\')\n                points = points.reshape(-1, 2)\n                if np.any(points < 0) or np.any(points > [width, height]):\n                    raise ValueError(f\'Out-of-bounds polygon {annotation["id"]}\')\n                lines.append("0 " + " ".join(\n                    f"{value:.8f}" for value in (points / [width, height]).ravel()))\n            (labels / f"{stem}.txt").write_text("\\n".join(lines) + ("\\n" if lines else ""))\n    path = data / "data.yaml"\n    path.write_text(yaml.safe_dump(dict(\n        path=str(data.resolve()), train="images/train", val="images/val",\n        names={0: "filament"},\n    )))\n    return path\n\n\ndef train_detector(ns, plan, shared_plan):\n    directory = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}"\n    directory.mkdir(parents=True, exist_ok=True)\n    identity = dict(plan_sha256=cv_digest(shared_plan), fold=plan["fold"])\n    require_identity(directory / "fold_identity.json", identity)\n    checkpoint, provenance = directory / "yolo_best.pt", directory / "yolo_provenance.json"\n    if provenance.exists():\n        saved = json.loads(provenance.read_text())\n        if saved["identity"] != identity or saved["sha256"] != file_hash(checkpoint):\n            raise ValueError("Saved detector checkpoint provenance mismatch.")\n        print(f"Fold {plan[\'fold\']}: reusing verified detector", flush=True)\n        return YOLO(str(checkpoint))\n    if checkpoint.exists():\n        raise ValueError("Checkpoint publication was interrupted; preserve it and inspect provenance.")\n    if not torch.cuda.is_available() and not ns["SMOKE_TEST"]:\n        raise RuntimeError("Enable a GPU before full training.")\n    settings = ns["V7"]\n    data = prepare_dataset(ns, plan, directory)\n    run = directory / "detector_runs" / "yolo"\n    last, best = run / "weights" / "last.pt", run / "weights" / "best.pt"\n    completed = False\n    if last.exists():\n        # Only read checkpoints generated by this identity-checked local run.\n        saved = torch.load(last, map_location="cpu", weights_only=False)\n        completed = saved["epoch"] < 0 or saved["epoch"] + 1 >= settings["epochs"]\n        del saved\n    started = time.monotonic()\n    if not completed:\n        if last.exists():\n            print(f"Fold {plan[\'fold\']}: resuming last completed YOLO epoch", flush=True)\n            model = YOLO(str(last))\n            model.train(resume=True, device=ns["YOLO_TRAIN_DEVICE"], workers=settings["workers"])\n        else:\n            print(f"Fold {plan[\'fold\']}: starting YOLO (no saved epoch)", flush=True)\n            model = YOLO(settings["weights"])\n            model.train(\n                data=str(data), epochs=settings["epochs"], imgsz=settings["image_size"],\n                batch=settings["batch_size"], device=ns["YOLO_TRAIN_DEVICE"],\n                workers=settings["workers"], patience=settings["patience"], time=None,\n                optimizer="AdamW", lr0=settings["learning_rate"], momentum=0.9,\n                weight_decay=0.0005, warmup_bias_lr=0.0, seed=ns["CFG"]["seed"],\n                single_cls=True, mask_ratio=2, overlap_mask=False,\n                hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, fliplr=0.5, flipud=0.0,\n                mosaic=0.3, degrees=10.0, scale=0.3, close_mosaic=5,\n                cache=False, save=True, save_period=1, plots=False,\n                project=str(run.parent), name=run.name, exist_ok=True,\n            )\n        del model\n        gc.collect()\n    if not best.is_file():\n        raise FileNotFoundError(f"No best detector checkpoint: {best}")\n    temporary = checkpoint.with_suffix(".tmp.pt")\n    shutil.copyfile(best, temporary)\n    temporary.replace(checkpoint)\n    atomic_json(provenance, dict(\n        identity=identity, sha256=file_hash(checkpoint), settings=settings,\n        latest_invocation_seconds=time.monotonic() - started,\n        note="Invocation time excludes earlier interrupted attempts; see YOLO results.csv.",\n    ))\n    return YOLO(str(checkpoint))\n\n\ndef encode_masks(instances):\n    return [dict(size=list(rle["size"]), counts=rle["counts"].decode("ascii"))\n            for rle in instances]\n\n\ndef decode_masks(items):\n    return [dict(size=item["size"], counts=item["counts"].encode("ascii")) for item in items]\n\n\ndef image_manifest(paths):\n    if not paths or len({p.stem for p in paths}) != len(paths):\n        raise ValueError("Prediction image IDs must be nonempty and unique.")\n    return [dict(name=p.name, sha256=file_hash(p)) for p in paths]\n\n\ndef predict_cached(ns, detector, paths, parameters, cache_path, identity):\n    """Save each completed inference batch; restart only missing images."""\n    paths = list(paths)\n    shape = ns["FULL_SHAPE"]\n    expected = {p.stem for p in paths}\n    cache_identity = dict(\n        model=identity, parameters=parameters,\n        images_sha256=cv_digest(image_manifest(paths)), shape=list(shape),\n    )\n    cache_path = Path(cache_path)\n    if cache_path.exists():\n        cache = json.loads(cache_path.read_text())\n        if cache["identity"] != cache_identity:\n            raise ValueError(f"Stale prediction cache: {cache_path}")\n    else:\n        cache = dict(identity=cache_identity, predictions={})\n    predictions = cache["predictions"]\n    if not set(predictions) <= expected:\n        raise ValueError("Prediction cache contains unknown image IDs.")\n    for items in predictions.values():\n        validate_disjoint(decode_masks(items), shape)\n    missing = [p for p in paths if p.stem not in predictions]\n    batch_size = ns["V7"]["inference_batch"]\n    if type(batch_size) is not int or batch_size < 1:\n        raise ValueError("inference_batch must be a positive integer.")\n    print(f"{cache_path.name}: {len(predictions)}/{len(paths)} saved", flush=True)\n    for start in range(0, len(missing), batch_size):\n        batch = missing[start:start + batch_size]\n        results = detector.predict(\n            source=[str(p) for p in batch], batch=batch_size,\n            imgsz=ns["V7"]["image_size"], conf=parameters["confidence"],\n            device=ns["YOLO_INFERENCE_DEVICE"], agnostic_nms=True, retina_masks=True,\n            verbose=False, stream=False,\n        )\n        if len(results) != len(batch):\n            raise ValueError("Detector returned an unexpected batch length.")\n        for path, result in zip(batch, results):\n            if Path(result.path).resolve() != path.resolve() or tuple(result.orig_shape) != tuple(shape):\n                raise ValueError("Detector result path/dimensions differ from its input.")\n            instances = []\n            if result.masks is not None:\n                masks = result.masks.data.cpu().numpy()\n                if masks.ndim != 3 or tuple(masks.shape[1:]) != tuple(shape):\n                    raise ValueError("Detector masks must retain original image dimensions.")\n                if not np.isin(masks, [0, 1]).all():\n                    raise ValueError("Detector masks must be finite and binary.")\n                instances = [mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))\n                             for mask in masks if mask.any()]\n            instances, _ = disjoint_instances(instances, shape, parameters["min_area"])\n            validate_disjoint(instances, shape)\n            predictions[path.stem] = encode_masks(instances)\n        atomic_json(cache_path, cache)\n        print(f"{cache_path.name}: {len(predictions)}/{len(paths)} saved", flush=True)\n    if set(predictions) != expected:\n        raise ValueError("Incomplete inference coverage.")\n    return predictions\n\n\ndef score_predictions(ns, names, predictions):\n    rows = []\n    for name in names:\n        instances = decode_masks(predictions[Path(name).stem])\n        scores = [ns["local_scores"](instances, ns["gt_rles"](record["id"]))\n                  for record in ns["records_by_name"][name]]\n        if not scores:\n            raise ValueError(f"No annotator sets: {name}")\n        rows.append(dict(\n            filename=name, annotator_sets=len(scores), instances=len(instances),\n            **{key: float(np.mean([score[key] for score in scores])) for key in scores[0]},\n        ))\n    if not rows:\n        raise ValueError("No evaluation images.")\n    means = {key: float(np.mean([row[key] for row in rows]))\n             for key in rows[0] if key not in ["filename", "annotator_sets"]}\n    return means, rows\n\n\ndef calibrate(ns, detector, plan, identity):\n    directory = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}"\n    path = directory / "calibration.json"\n    if path.exists():\n        saved = json.loads(path.read_text())\n        if saved["identity"] != identity:\n            raise ValueError("Calibration provenance mismatch.")\n        return saved\n    names = plan["splits"]["tune"][:ns["V7"]["tuning_images"]]\n    trials = []\n    for confidence in ns["V7"]["confidences"]:\n        for min_area in ns["V7"]["min_areas"]:\n            parameters = dict(mode="yolo", confidence=confidence, min_area=min_area)\n            predictions = predict_cached(\n                ns, detector, [ns["train_lookup"][name] for name in names], parameters,\n                directory / f"tune_{confidence}_{min_area}.json", identity,\n            )\n            means, _ = score_predictions(ns, names, predictions)\n            trials.append(dict(parameters=parameters, scores=means))\n    selected = max(trials, key=lambda trial: trial["scores"]["pq"])\n    result = dict(identity=identity, selected=selected, trials=trials, tuning_names=names)\n    atomic_json(path, result)\n    print("TUNING ONLY, not independent performance:", selected, flush=True)\n    return result\n\n\ndef fold_model_identity(ns, plan, shared_plan):\n    checkpoint = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}" / "yolo_best.pt"\n    return dict(plan_sha256=cv_digest(shared_plan), fold=plan["fold"],\n                checkpoint_sha256=file_hash(checkpoint))\n\n\ndef run_fold(ns, plan, shared_plan):\n    directory = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}"\n    path = directory / "fold_result.json"\n    if path.exists():\n        result = json.loads(path.read_text())\n        validate_fold_result(result, plan, cv_digest(shared_plan))\n        identity = fold_model_identity(ns, plan, shared_plan)\n        if result["checkpoint_hashes"] != {"yolo_best.pt": identity["checkpoint_sha256"]}:\n            raise ValueError("Completed fold checkpoint changed.")\n        print(f"Fold {plan[\'fold\']}: reusing completed outer evaluation", flush=True)\n        return result\n    detector = train_detector(ns, plan, shared_plan)\n    try:\n        identity = fold_model_identity(ns, plan, shared_plan)\n        calibration = calibrate(ns, detector, plan, identity)\n        parameters = calibration["selected"]["parameters"]\n        names = plan["splits"]["holdout"]\n        predictions = predict_cached(\n            ns, detector, [ns["train_lookup"][name] for name in names], parameters,\n            directory / "outer_predictions.json", identity,\n        )\n        means, rows = score_predictions(ns, names, predictions)\n        result = dict(\n            fold=plan["fold"], plan_sha256=cv_digest(shared_plan), status="complete",\n            smoke_test=ns["SMOKE_TEST"],\n            checkpoint_hashes={"yolo_best.pt": identity["checkpoint_sha256"]},\n            selected_parameters=parameters, inner_selected_scores=calibration["selected"]["scores"],\n            outer_mean=means, outer_rows=rows,\n        )\n        validate_fold_result(result, plan, cv_digest(shared_plan))\n        atomic_json(path, result)\n        return result\n    finally:\n        del detector\n        gc.collect()\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n\ndef write_submission(ns, shared_plan, predictions, results):\n    """Validate the exact CSV read back from disk and publish its checksum last."""\n    if ns["SMOKE_TEST"] or shared_plan["smoke_test"]:\n        raise ValueError("Smoke experiments cannot export competition submissions.")\n    root, shape = ns["CV_ROOT"], ns["FULL_SHAPE"]\n    paths = ns["test_paths"]\n    names = {path.stem for path in paths}\n    if len(predictions) != 5 or any(set(fold) != names for fold in predictions):\n        raise ValueError("Export requires complete test coverage from exactly five models.")\n    rows = []\n    for index, path in enumerate(paths):\n        per_fold = [decode_masks(fold[path.stem]) for fold in predictions]\n        instances = fuse_fold_instances(per_fold, shape)\n        instances, _ = disjoint_instances(instances, shape, ENSEMBLE_POLICY["min_area"])\n        validate_disjoint(instances, shape)\n        rows.extend(dict(filament_id=f"{path.stem}_{i}", segmentation_rle=rle["counts"].decode("ascii"))\n                    for i, rle in enumerate(instances, 1))\n        print(f"Fusion: {index + 1}/{len(paths)}", flush=True)\n    validate_submission_rows(rows, names, shape)\n    destination = root / "submission_V7_yolo_only.csv"\n    temporary = destination.with_suffix(".tmp.csv")\n    with temporary.open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=SUBMISSION_FIELDS)\n        writer.writeheader()\n        writer.writerows(rows)\n    with temporary.open(newline="") as stream:\n        reader = csv.DictReader(stream)\n        if reader.fieldnames != SUBMISSION_FIELDS:\n            raise ValueError("Written CSV columns differ.")\n        readback = list(reader)\n    if readback != rows:\n        raise ValueError("Written CSV differs from generated predictions.")\n    audit = validate_submission_rows(readback, names, shape)\n    digest = file_hash(temporary)\n    temporary.replace(destination)\n    atomic_json(root / "ensemble_audit.json", dict(\n        run_name=root.name, architecture="YOLO-only; no crop refiner",\n        plan_sha256=cv_digest(shared_plan), policy=ENSEMBLE_POLICY,\n        test_manifest_sha256=cv_digest(image_manifest(paths)),\n        folds=[dict(fold=r["fold"], hashes=r["checkpoint_hashes"],\n                    parameters=r["selected_parameters"]) for r in results],\n        validation=audit, submission_filename=destination.name, submission_sha256=digest,\n    ))\n    if file_hash(destination) != digest:\n        raise ValueError("Final submission checksum mismatch.")\n    print("FINAL V7 CSV VERIFIED:", destination, "\\nSHA256:", digest, "\\nVALIDATION:", audit,\n          "\\nNo upload performed. Match this hash before manual submission.", flush=True)\n    return destination\n\n\ndef export_ensemble(ns, plans, shared_plan):\n    if ns["SMOKE_TEST"] or shared_plan["smoke_test"]:\n        raise ValueError("Smoke experiments cannot export competition submissions.")\n    if sorted(plan["fold"] for plan in plans) != list(range(5)):\n        raise ValueError("Five unique fold plans are required.")\n    plans = sorted(plans, key=lambda plan: plan["fold"])\n    results, predictions = [], []\n    for plan in plans:\n        directory = ns["CV_ROOT"] / f"fold_{plan[\'fold\']}"\n        result = json.loads((directory / "fold_result.json").read_text())\n        validate_fold_result(result, plan, cv_digest(shared_plan))\n        identity = fold_model_identity(ns, plan, shared_plan)\n        if result["checkpoint_hashes"] != {"yolo_best.pt": identity["checkpoint_sha256"]}:\n            raise ValueError("Export checkpoint differs from evaluated model.")\n        calibration = json.loads((directory / "calibration.json").read_text())\n        if (calibration["identity"] != identity\n                or calibration["selected"]["parameters"] != result["selected_parameters"]):\n            raise ValueError("Export calibration differs from evaluated settings.")\n        detector = YOLO(str(directory / "yolo_best.pt"))\n        try:\n            predictions.append(predict_cached(\n                ns, detector, ns["test_paths"], result["selected_parameters"],\n                directory / "test_predictions.json", identity,\n            ))\n            results.append(result)\n        finally:\n            del detector\n            gc.collect()\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n    return write_submission(ns, shared_plan, predictions, results)\n')
if str(runtime_dir) not in sys.path:
    sys.path.insert(0, str(runtime_dir))
from grouped_yolo_cv import (
    ENSEMBLE_POLICY, cv_digest, fold_records, make_day_folds,
    collect_fold_results, summarize_folds,
)
from v7_yolo_runtime import run_fold, export_ensemble


## Five day-grouped folds
Each image day is isolated to train, tuning, or outer holdout within each fold. All five folds are selected by default; their holdout predictions are used once for the grouped outer evaluation.


In [ ]:
if (not FOLDS_TO_RUN or len(set(FOLDS_TO_RUN)) != len(FOLDS_TO_RUN)
        or any(type(fold) is not int or fold not in range(5) for fold in FOLDS_TO_RUN)):
    raise ValueError("Choose distinct integer fold indices from 0 through 4.")
cv_plans = make_day_folds(records_by_name, CFG["seed"], inner_fraction=INNER_TUNE_FRACTION)
cv_plan = dict(
    name=RUN_NAME, seed=CFG["seed"], annotation_sha256=annotation_sha256,
    settings=V7, inner_tune_fraction=INNER_TUNE_FRACTION,
    folds=cv_plans, smoke_test=SMOKE_TEST, ensemble_policy=ENSEMBLE_POLICY,
    architecture="YOLO-only; no crop refiner",
    runtime_sha256=hashlib.sha256((runtime_dir / "v7_yolo_runtime.py").read_bytes()).hexdigest(),
)
plan_path = CV_ROOT / "cv_plan.json"
if plan_path.exists() and json.loads(plan_path.read_text()) != cv_plan:
    raise ValueError("Existing V7 run plan differs. Choose a new RUN_NAME.")
if not plan_path.exists():
    plan_path.write_text(json.dumps(cv_plan, indent=2, allow_nan=False))
cv_plan_hash = cv_digest(cv_plan)
for fold_plan in cv_plans:
    entries = fold_records(fold_plan, records_by_name)
    print("Fold", fold_plan["fold"], {
        phase: dict(physical_images=len(fold_plan["splits"][phase]),
                    annotator_sets=len(selected),
                    instances=sum(len(annotations_by_id[r["id"]]) for r in selected))
        for phase, selected in entries.items()
    })
print("Selected folds:", FOLDS_TO_RUN, "Plan SHA256:", cv_plan_hash)
with (CV_ROOT / "requirements-actual.txt").open("w") as stream:
    subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze"],
                   stdout=stream, check=True)


## Train and evaluate
Run all five folds sequentially. Training is bounded by the 10-epoch maximum, not by elapsed time. Each fold resumes from its own saved YOLO checkpoint if interrupted.


In [ ]:
cv_fold_results = []
for fold in FOLDS_TO_RUN:
    result = run_fold(globals(), cv_plans[fold], cv_plan)
    cv_fold_results.append(result)
    print("OUTER FOLD", fold, result["outer_mean"], flush=True)


## Review grouped results
The full five-fold aggregate is shown only when all five matching fold results are available. Partial runs are reported as incomplete.


In [ ]:
cv_results = collect_fold_results([CV_ROOT, *map(Path, COLLECT_ROOTS)], cv_plans, cv_plan_hash)
cv_summary = summarize_folds(cv_results, cv_plans, cv_plan_hash)
print(json.dumps(cv_summary, indent=2, allow_nan=False))


## Prepare and submit the five-model CSV
Run only after reviewing the completed fold summary. The export validates disjoint masks, CSV readback, image coverage, and the final checksum. The next cell submits the verified file through the Kaggle CLI; configure Kaggle API credentials first.


In [ ]:
if not cv_summary["complete"]:
    raise RuntimeError("All five folds must finish before ensemble submission export.")
submission_path = export_ensemble(globals(), cv_plans, cv_plan)


In [ ]:
if not cv_summary["complete"] or not submission_path.is_file():
    raise RuntimeError("A complete five-fold run and verified CSV are required before submission.")
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
subprocess.run([
    "kaggle", "competitions", "submit", "-c", COMPETITION,
    "-f", str(submission_path), "-m", "V7: five-fold YOLO-only, 10-epoch maximum",
], check=True)
